**Programación para código**

In [ ]:
import pandas as pd
!pip install pyreadstat
import pyreadstat
import os # manejo de ruta de las carpetas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.8 MB/s eta 0:00:00


**Configurar acceso a base en Google Drive**

Conectado directamente al acceso directo de mi drive en línea. Le compartí la información desde el correo del observatorio a mi correo de FLACSO, luego creé un acceso directo en mi unidad para trabajar todo en línea. Esto se puede replicar para cualquiera que repita el proceso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Creación de matriz para rellenar**

**Generar Excel**

In [ ]:
import unicodedata
# --- 1. CONFIGURACIÓN ---
# Esta es la ruta basada en tu captura de pantalla
path_base = "/content/drive/MyDrive/Observatorio/Bases/ENEMDU/Originales/Diciembres"

var_buscadas = {
    "Genero": ["sexo", "p02"], #p02 en 2007
    "Edad": ["edad", "p03"], #p03 en 2007
    "Educación": ["nivinst", "p10a", "p10b"],# p10a p10b desde 2007
    "Estado civil": ["p06", "p07", "civil", "conyugal", "pe4a","pe13c"],
    "Provincia": ["ciudad", "prov"], #prov desde 2007 en adelante solo prov
    "Establecimiento tiene RUC": ["p49","pe51", "pe49", "pe51"],
    "Informal-Afiliación": ["iess", "p05a", "p05b"], #p05a p05b desde 2007?
    "Tipo de contrato": ["sitios", "p43", "estabil"], #p43 desde 2007 ?
    "Informal-Subempleo": ["condact", "CONDACT", "p36"], #p36 desde 2007 ?
    "Informal-Ocupación": ["condina", "p42"], #p42 desde 2007?
    "Informal-Sector informal": ["peamsiu", "secemp"], #desde 2007?
    "Ciudades": ["ciudad"] #desde 2007?
}
#desde 2001 hay etnia
def limpiar_texto(t):
    if not t: return ""
    return ''.join(c for c in unicodedata.normalize('NFD', str(t))
                  if unicodedata.category(c) != 'Mn').lower()

anios = range(1990, 2025)
matriz = pd.DataFrame(index=var_buscadas.keys(), columns=anios)

# --- 2. BUCLE DE PROCESAMIENTO ---
for anio in anios:
    # Determinamos la subcarpeta según el año
    if 1990 <= anio <= 1999: sub_c = "1990-1999"
    elif 2000 <= anio <= 2006: sub_c = "2000-2006"
    elif 2007 <= anio <= 2017: sub_c = "2007-2017"
    else: sub_c = os.path.join("2018-presente", "Mensuales")

    archivo = f"empleo{anio}.dta"
    ruta_completa = os.path.join(path_base, sub_c, archivo)

    print(f"🔎 Buscando {anio} en {sub_c}...", end=" ")

    if os.path.exists(ruta_completa):
        try:
            # Leemos solo metadatos para que sea veloz
            _, meta = pyreadstat.read_dta(ruta_completa, metadataonly=True)

            for concepto, lista_nombres in var_buscadas.items():
                encontradas = []

                # Ahora iteramos sobre la lista oficial de nombres de variables en el .dta
                for var_name in meta.column_names:
                    # Si el nombre de la variable (en minúsculas) coincide con tu lista
                    if var_name.lower() in [n.lower() for n in lista_nombres]:

                        # Obtenemos la etiqueta para saber qué es (ej: "P03: Edad")
                        label = meta.column_names_to_labels.get(var_name, "Sin etiqueta")

                        # Extraemos categorías (1=Hombre, etc.)
                        dic_labels = meta.variable_value_labels.get(var_name, {})
                        if dic_labels:
                            txt_lab = " / ".join([f"{str(k)} {v}" for k, v in dic_labels.items()])
                            res = f"{var_name} ({label}): {txt_lab}"
                        else:
                            res = f"{var_name} ({label}): (Continua)"

                        encontradas.append(res)

                matriz.at[concepto, anio] = " | ".join(encontradas) if encontradas else "NO ENCONTRADA"
            print("✅")
        except Exception as e:
            print(f"⚠️ Error al leer o procesar: {e}")
    else:
        # Pista: si no lo encuentra, imprimimos la ruta para ver qué falló
        print(f"❌ No existe en: {ruta_completa}")

# --- 3. GUARDAR RESULTADO ---
nombre_archivo = "Matriz_ENEMDU_Consolidad.xlsx"
matriz.to_excel(nombre_archivo)
print(f"\n🥳 ¡TERMINADO! Descarga '{nombre_archivo}' del menú lateral de Colab.")

🔎 Buscando 1990 en 1990-1999... ✅
🔎 Buscando 1991 en 1990-1999... ✅
🔎 Buscando 1992 en 1990-1999... ✅
🔎 Buscando 1993 en 1990-1999... ✅
🔎 Buscando 1994 en 1990-1999... ✅
🔎 Buscando 1995 en 1990-1999... ✅
🔎 Buscando 1996 en 1990-1999... ✅
🔎 Buscando 1997 en 1990-1999... ✅
🔎 Buscando 1998 en 1990-1999... ✅
🔎 Buscando 1999 en 1990-1999... ✅
🔎 Buscando 2000 en 2000-2006... ✅
🔎 Buscando 2001 en 2000-2006... ✅
🔎 Buscando 2002 en 2000-2006... ✅
🔎 Buscando 2003 en 2000-2006... ✅
🔎 Buscando 2004 en 2000-2006... ✅
🔎 Buscando 2005 en 2000-2006... ✅
🔎 Buscando 2006 en 2000-2006... ✅
🔎 Buscando 2007 en 2007-2017... ✅
🔎 Buscando 2008 en 2007-2017... ✅
🔎 Buscando 2009 en 2007-2017... ✅
🔎 Buscando 2010 en 2007-2017... ✅
🔎 Buscando 2011 en 2007-2017... ✅
🔎 Buscando 2012 en 2007-2017... ✅
🔎 Buscando 2013 en 2007-2017... ✅
🔎 Buscando 2014 en 2007-2017... ✅
🔎 Buscando 2015 en 2007-2017... ✅
🔎 Buscando 2016 en 2007-2017... ✅
🔎 Buscando 2017 en 2007-2017... ✅
🔎 Buscando 2018 en 2018-presente/Mensuales... ✅


In [ ]:
import pandas as pd
base = pd.read_excel('Matriz_ENEMDU_Wil_Final.xlsx')
print(base.head(10))

                  Unnamed: 0  \
0                     Genero   
1                       Edad   
2                  Educación   
3               Estado civil   
4                  Provincia   
5  Establecimiento tiene RUC   
6           Seguridad social   
7           Tipo de contrato   
8         Informal-Subempleo   
9        Informal-Afiliación   

                                                1990  \
0                           sexo: 1 hombre / 2 mujer   
1  edad: 0 menos de un año / 98 98 y màs / 99 no ...   
2  nivinst: 1 ninguno / 2 curso de alfabetización...   
3                                             MANUAL   
4                                             MANUAL   
5  nivinst: 1 ninguno / 2 curso de alfabetización...   
6                                  iess: 1 si / 2 no   
7                                             MANUAL   
8                                             MANUAL   
9                                  iess: 1 si / 2 no   

                              

**respaldo**

In [ ]:
import unicodedata
# --- 1. CONFIGURACIÓN ---
# Esta es la ruta basada en tu captura de pantalla
path_base = "/content/drive/MyDrive/Observatorio/Bases/ENEMDU/Originales/Diciembres"

var_buscadas = {
    "Genero": ["sexo", "p02"], #p02 en 2007
    "Edad": ["edad", "p03"], #p03 en 2007
    "Educación": ["nivinst", "p10a", "p10b"],# p10a p10b desde 2007
    "Estado civil": ["p06", "p07", "civil", "conyugal"], #p06 en 2007
    "Provincia": ["ciudad", "prov"], #prov desde 2007 en adelante solo prov
    "Establecimiento tiene RUC": ["p49"], #aparece en 2007?
    "Informal-Afiliación": ["iess", "p05a", "p05b"], #p05a p05b desde 2007?
    "Tipo de contrato": ["sitios", "p43"], #p43 desde 2007 ?
    "Informal-Subempleo": ["condact", "CONDACT", "p36"], #p36 desde 2007 ?
    "Informal-Ocupación": ["condina", "p42"], #p42 desde 2007?
    "Informal-Sector informal": ["peamsiu", "secemp"], #desde 2007?
    "Ciudades": ["ciudad"] #desde 2007?
}

def limpiar_texto(t):
    if not t: return ""
    return ''.join(c for c in unicodedata.normalize('NFD', str(t))
                  if unicodedata.category(c) != 'Mn').lower()

anios = range(1990, 2025)
matriz = pd.DataFrame(index=var_buscadas.keys(), columns=anios)

# --- 2. BUCLE DE PROCESAMIENTO ---
for anio in anios:
    # Determinamos la subcarpeta según el año
    if 1990 <= anio <= 1999: sub_c = "1990-1999"
    elif 2000 <= anio <= 2006: sub_c = "2000-2006"
    elif 2007 <= anio <= 2017: sub_c = "2007-2017"
    else: sub_c = os.path.join("2018-presente", "Mensuales")

    archivo = f"empleo{anio}.dta"
    ruta_completa = os.path.join(path_base, sub_c, archivo)

    print(f"🔎 Buscando {anio} en {sub_c}...", end=" ")

    if os.path.exists(ruta_completa):
        try:
            # Leemos solo metadatos para que sea veloz
            _, meta = pyreadstat.read_dta(ruta_completa, metadataonly=True)

            for concepto, palabras_clave in var_buscadas.items():
                encontradas = []
                for var_name, label in meta.column_names_to_labels.items():
                    label_limpia = limpiar_texto(label)
                    if any(p.lower() in label_limpia for p in palabras_clave):
                        dic_labels = meta.variable_value_labels.get(var_name, {})
                        if dic_labels:
                            txt_lab = " / ".join([f"{str(k)} {v}" for k, v in dic_labels.items()])
                            res = f"{var_name}: {txt_lab}"
                        else:
                            res = f"{var_name}: (Continua)"
                        encontradas.append(res)

                matriz.at[concepto, anio] = " | ".join(encontradas) if encontradas else "MANUAL"
            print("✅")
        except Exception as e:
            print(f"⚠️ Error al leer: {e}")
    else:
        # Pista: si no lo encuentra, imprimimos la ruta para ver qué falló
        print(f"❌ No existe en: {ruta_completa}")

# --- 3. GUARDAR RESULTADO ---
nombre_archivo = "Matriz_ENEMDU_Wil_Final.xlsx"
matriz.to_excel(nombre_archivo)
print(f"\n🥳 ¡TERMINADO! Descarga '{nombre_archivo}' del menú lateral de Colab.")

🔎 Buscando 1990 en 1990-1999... ✅
🔎 Buscando 1991 en 1990-1999... ✅
🔎 Buscando 1992 en 1990-1999... ✅
🔎 Buscando 1993 en 1990-1999... ✅
🔎 Buscando 1994 en 1990-1999... ✅
🔎 Buscando 1995 en 1990-1999... ✅
🔎 Buscando 1996 en 1990-1999... ✅
🔎 Buscando 1997 en 1990-1999... ✅
🔎 Buscando 1998 en 1990-1999... ✅
🔎 Buscando 1999 en 1990-1999... ✅
🔎 Buscando 2000 en 2000-2006... ✅
🔎 Buscando 2001 en 2000-2006... ✅
🔎 Buscando 2002 en 2000-2006... ✅
🔎 Buscando 2003 en 2000-2006... ✅
🔎 Buscando 2004 en 2000-2006... ✅
🔎 Buscando 2005 en 2000-2006... ✅
🔎 Buscando 2006 en 2000-2006... ✅
🔎 Buscando 2007 en 2007-2017... ✅
🔎 Buscando 2008 en 2007-2017... ✅
🔎 Buscando 2009 en 2007-2017... ✅
🔎 Buscando 2010 en 2007-2017... ✅
🔎 Buscando 2011 en 2007-2017... ✅
🔎 Buscando 2012 en 2007-2017... ✅
🔎 Buscando 2013 en 2007-2017... ✅
🔎 Buscando 2014 en 2007-2017... ✅
🔎 Buscando 2015 en 2007-2017... ✅
🔎 Buscando 2016 en 2007-2017... ✅
🔎 Buscando 2017 en 2007-2017... ✅
🔎 Buscando 2018 en 2018-presente/Mensuales... ✅
